# The RSA Cryptosystem
**Category: Computer Science**

## 1. Introduction

RSA (Rivest–Shamir–Adleman, 1977) was the first practical public-key cryptosystem. Its security rests on the **integer factorisation problem**: given a large composite number $n = pq$, recovering $p$ and $q$ is computationally infeasible for suitably large primes. RSA underpins TLS/HTTPS, SSH, and digital signatures across the internet.

## 2. Mathematical Foundations

### 2.1 Euler's Theorem

For integers $a$ and $n$ with $\gcd(a, n) = 1$, Euler's theorem states
$$
a^{\phi(n)} \equiv 1 \pmod{n}
$$
where $\phi(n)$ is Euler's totient function. For $n = pq$ with distinct primes $p, q$:
$$
\phi(n) = (p-1)(q-1)
$$
This identity is the cornerstone of RSA correctness.

### 2.2 Key Generation

1. Choose two large distinct primes $p$ and $q$. Set $n = pq$.
2. Compute $\phi(n) = (p-1)(q-1)$.
3. Choose $e$ with $1 < e < \phi(n)$ and $\gcd(e, \phi(n)) = 1$. Common choice: $e = 65537 = 2^{16} + 1$.
4. Compute $d \equiv e^{-1} \pmod{\phi(n)}$ via the extended Euclidean algorithm.
5. **Public key**: $(n, e)$. **Private key**: $(n, d)$.

### 2.3 Encryption and Decryption

For a message $m$ with $0 < m < n$:
$$
\text{Encrypt:} \quad c \equiv m^e \pmod{n}
$$
$$
\text{Decrypt:} \quad m \equiv c^d \pmod{n}
$$
**Correctness**: Since $ed \equiv 1 \pmod{\phi(n)}$, there exists $k$ such that $ed = 1 + k\phi(n)$, and by Euler's theorem
$$
c^d = m^{ed} = m^{1 + k\phi(n)} = m \cdot (m^{\phi(n)})^k \equiv m \cdot 1^k = m \pmod{n}
$$

## 3. Security Considerations

### 3.1 Key Size

The best known algorithm for factoring a $k$-bit integer is the **General Number Field Sieve** (GNFS), with sub-exponential running time
$$
\exp\!\left(\left(\frac{64}{9}\right)^{1/3} (\ln n)^{1/3} (\ln \ln n)^{2/3}\right)
$$
NIST recommends $n \geq 2048$ bits for security beyond 2030, corresponding to roughly $112$ bits of symmetric-equivalent security. A $4096$-bit key gives $\approx 140$ bits of security.

### 3.2 Common Pitfalls

- **Small $e$ with small $m$**: if $m^e < n$, the modular reduction never activates and $m = c^{1/e}$ over the integers.
- **Common modulus attack**: if two users share $n$ but use coprime exponents $e_1, e_2$, Bezout's identity yields $m$ from $c_1, c_2$.
- **No padding**: textbook RSA is deterministic and malleable. In practice, **OAEP** (Optimal Asymmetric Encryption Padding) is mandatory.
- **Quantum threat**: Shor's algorithm factors $n$ in polynomial time on a quantum computer, breaking RSA entirely.

## 4. Numerical Demonstration

### 4.1 Toy RSA Implementation

In [ ]:
import math
import random

def extended_gcd(a, b):
    if b == 0:
        return a, 1, 0
    g, x, y = extended_gcd(b, a % b)
    return g, y, x - (a // b) * y

def mod_inverse(e, phi):
    g, x, _ = extended_gcd(e, phi)
    if g != 1:
        raise ValueError('Inverse does not exist')
    return x % phi

def is_prime(n, k=20):
    """Miller-Rabin primality test."""
    if n < 2: return False
    if n == 2: return True
    if n % 2 == 0: return False
    r, d = 0, n - 1
    while d % 2 == 0:
        r += 1; d //= 2
    for _ in range(k):
        a = random.randrange(2, n - 1)
        x = pow(a, d, n)
        if x in (1, n - 1): continue
        for _ in range(r - 1):
            x = pow(x, 2, n)
            if x == n - 1: break
        else:
            return False
    return True

# Small primes for demonstration only
p, q = 61, 53
n = p * q
phi_n = (p - 1) * (q - 1)
e = 17
d = mod_inverse(e, phi_n)

print(f'n = {n},  phi(n) = {phi_n}')
print(f'Public key:  (n={n}, e={e})')
print(f'Private key: (n={n}, d={d})')

message = 42
ciphertext = pow(message, e, n)
decrypted  = pow(ciphertext, d, n)
print(f'\nMessage:    {message}')
print(f'Ciphertext: {ciphertext}')
print(f'Decrypted:  {decrypted}')

### 4.2 GNFS Security Estimate vs. Key Size

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

bits = np.linspace(512, 8192, 500)
ln2 = np.log(2)

# GNFS complexity ~ exp((64/9)^(1/3) * (k*ln2)^(1/3) * (k*ln2*ln(k*ln2))^(2/3) )
# Equivalent symmetric security in bits ~ log2 of complexity
def gnfs_security(k):
    ln_n = k * ln2
    exponent = (64/9)**(1/3) * ln_n**(1/3) * (np.log(ln_n))**(2/3)
    return exponent / ln2  # bits of security

sec = gnfs_security(bits)

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(bits, sec, linewidth=2)
ax.axhline(112, color='orange', linestyle='--', label='112-bit security (NIST 2030+)')
ax.axhline(128, color='green',  linestyle='--', label='128-bit security')
ax.axvline(2048, color='orange', linestyle=':')
ax.axvline(3072, color='green',  linestyle=':')
ax.set_xlabel('RSA key size (bits)')
ax.set_ylabel('Equivalent symmetric security (bits)')
ax.set_title('RSA Security vs. Key Size (GNFS estimate)')
ax.legend()
plt.tight_layout()
plt.show()